In [1]:
%load_ext autoreload
%autoreload 2
# conda activate fishyrl
# tensorboard --logdir ./examples/runs

TODO Ordered List
- Implement checkpointing
- Add timers
- Refactor initialization
- Test or revise discretization and continuous actions (don't forget about objective in `train`)
- Try MuJoCo
- Implement https://rlgym.org/
- Add mixed precision training, or at least fp16
- Implement distributed
- Add missing tests
- Vectorize discrete and discretized actions, as well as buffer storage and sampling

Novel changes
- TwoHot discretization
- Could add variation output to reward, allowing for trust regions and exploration, maybe an auxiliary loss?

In [2]:
import math
import os
import time

import torch
import torch.utils.tensorboard

import fishyrl as frl

In [3]:
# Tested
# CartPole-v1: Begins converging to optimal after 10 minutes w/ 512 EOD, 2 NB, 512 DD, 32 SD, 512 DD, and sequence length 10
# MountainCar-v0: Doesn't get an initial reward to work from with same params as CartPole-v1
# BipedalWalker-v3: Learns to stand
# https://github.com/Eclectic-Sheep/sheeprl/issues/229

In [ ]:
# Load config (NOTE: Times are unreliable here, as they were used in varying workloads)
# cfg = frl.utilities.load_config('../examples/CartPole.yaml', '../examples/Dreamer_Small.yaml')  # Load CartPole on top of Dreamer_Small (CartPole-v1_2026-04-10_19-57-31, 23k steps, 1.6h to full-optimal)
cfg = frl.utilities.load_config('../examples/LunarLander.yaml', '../examples/Dreamer_Small.yaml')  # (LunarLander-v3_2026-04-11_13-49-49, XXk steps, X.Xh to full-optimal)

# Load environment and actions
envs, actions = frl.dreamer.get_environments_and_actions(cfg=cfg)

# Construct models
world_model, actor_critic_model, utility_modules = frl.dreamer.construct_models(
    env_obs_dim=math.prod(envs.observation_space.shape[1:]),  # TODO: Revise for CNN
    env_actions=actions,
    device='cuda',
    cfg=cfg,
)

# Tensorboard writer
folder_name = f'{cfg.env.name}_{time.strftime("%Y-%m-%d_%H-%M-%S")}'
folder_name = f'./runs/{folder_name}'
writer = torch.utils.tensorboard.SummaryWriter(folder_name)

In [ ]:
# Train the model
frl.dreamer.train_loop(
    envs=envs,
    world_model=world_model,
    actor_critic_model=actor_critic_model,
    utility_modules=utility_modules,
    tensorboard_writer=writer,
    cfg=cfg,
)

In [ ]:
# Close tensorboard writer
writer.close()

In [ ]:
# Save model
frl.dreamer.save_models(
    path=os.path.join(folder_name, 'model.pth'),
    world_model=world_model,
    actor_critic_model=actor_critic_model,
    utility_modules=utility_modules,
)

# Load model
frl.dreamer.load_models(
    path=os.path.join(folder_name, 'model.pth'),
    world_model=world_model,
    actor_critic_model=actor_critic_model,
    utility_modules=utility_modules,
)